In [3]:
import hashlib
import pandas as pd
from itertools import combinations

# --- Config ---
FILES = [
    "evals_1k.csv",
    "deepmind_test_62k.csv",
    "test_2k.csv",
    "train_20k.csv",
]

In [8]:
# --- Load CSVs & gather FEN ID sets ---
dfs = {f: pd.read_csv(f, usecols=["FEN ID"]) for f in FILES}
fsets = {f: set(df["FEN ID"]) for f, df in dfs.items()}

# --- In‑file FEN ID duplicates ---
print("✦ In‑file duplicate counts (by FEN ID)")
for f, df in dfs.items():
    dup_count = len(df) - df["FEN ID"].nunique()
    print(f"  {f:<22} -> {dup_count}")

# --- Pairwise overlaps ---
print("\n✦ Pairwise FEN ID overlaps")
for (fa, fb) in combinations(FILES, 2):
    overlap = len(fsets[fa] & fsets[fb])
    print(f"  {fa:<22} ∩ {fb:<22} = {overlap}")

✦ In‑file duplicate counts (by FEN ID)
  evals_1k.csv           -> 0
  deepmind_test_62k.csv  -> 1
  test_2k.csv            -> 0
  train_20k.csv          -> 0

✦ Pairwise FEN ID overlaps
  evals_1k.csv           ∩ deepmind_test_62k.csv  = 1000
  evals_1k.csv           ∩ test_2k.csv            = 0
  evals_1k.csv           ∩ train_20k.csv          = 0
  deepmind_test_62k.csv  ∩ test_2k.csv            = 2000
  deepmind_test_62k.csv  ∩ train_20k.csv          = 20000
  test_2k.csv            ∩ train_20k.csv          = 0


In [6]:
# Build global FEN→ID map (6 digit alphanumeric)
def fen_to_uid(fen):
    # Take sha256 hash, use first 6 hex digits, convert to base36 (alphanumeric), pad to 6
    h = hashlib.sha256(fen.strip().encode()).hexdigest()[:8]  # Take 8 to have more entropy
    as_int = int(h, 16)
    chars = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    out = ''
    while as_int > 0 and len(out) < 6:
        out = chars[as_int % 36] + out
        as_int //= 36
    return out.rjust(6, '0')  # Pad to 6 chars

fen_to_id = {}
for fp in FILES:
    for fen in pd.read_csv(fp, usecols=["FEN"])["FEN"]:
        fen = str(fen).strip()
        if fen not in fen_to_id:
            fen_to_id[fen] = fen_to_uid(fen)

# Add “FEN ID” column and overwrite files
for fp in FILES:
    df = pd.read_csv(fp)
    df["FEN ID"] = df["FEN"].map(fen_to_id)
    df.to_csv(fp, index=False)